# Twitter Trending Local Profile

Local-only profiling notebook for validated Twitter trending snapshot artifacts.


## 1. Load And Validate Local Artifacts

This section enforces the blocker gate from the prior validation phase.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / "scripts" / "profile_twitter_trending.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repository root containing scripts/profile_twitter_trending.py"
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.profile_twitter_trending import build_profile_report, load_snapshot

SNAPSHOT_PATH = ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_full.parquet"
SNAPSHOT_METADATA_PATH = (
    ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_snapshot_metadata.json"
)
VALIDATION_REPORT_PATH = (
    ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_validation_report.json"
)
SAMPLE_PARQUET_PATH = ROOT / "data/samples/twitter_trending_sample_1000.parquet"
SAMPLE_CSV_PATH = ROOT / "data/samples/twitter_trending_sample_1000.csv"

required_paths = [
    SNAPSHOT_PATH,
    SNAPSHOT_METADATA_PATH,
    VALIDATION_REPORT_PATH,
    SAMPLE_PARQUET_PATH,
    SAMPLE_CSV_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required local artifacts are missing. Prior phase is incomplete. Missing: "
        + ", ".join(missing_paths)
    )

validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))
if validation_report.get("overall_pass") is not True:
    raise RuntimeError(
        "Local snapshot validation did not pass. "
        f"overall_pass={validation_report.get('overall_pass')!r}"
    )

blocking_issues = validation_report.get("blocking_issues") or []
if blocking_issues:
    raise RuntimeError(
        "Snapshot validation report has blocking issues: "
        + "; ".join(str(issue) for issue in blocking_issues)
    )

print("repo_root:", ROOT)
print("snapshot_path:", SNAPSHOT_PATH)
print("validation_report_path:", VALIDATION_REPORT_PATH)
print("validation_status: PASS")


repo_root: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject
snapshot_path: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/reference_snapshots/twitter_trending/twitter_trending_full.parquet
validation_report_path: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/reference_snapshots/twitter_trending/twitter_trending_validation_report.json
validation_status: PASS


In [2]:
df = load_snapshot(SNAPSHOT_PATH)
profile = build_profile_report(df)

print("rows:", profile["dataset"]["row_count"])
print("columns:", profile["dataset"]["columns"])


rows: 101731
columns: ['num_hours', 'date', 'name', 'counts']


## 2. Dataset Overview


In [3]:
overview_df = pd.DataFrame([
    {
        "row_count": profile["dataset"]["row_count"],
        "unique_topic_count": profile["dataset"]["unique_topic_count"],
        "unique_date_count": profile["dataset"]["unique_date_count"],
        "rows_per_day_min": profile["dataset"]["rows_per_day_summary"]["min"],
        "rows_per_day_max": profile["dataset"]["rows_per_day_summary"]["max"],
        "rows_per_day_median": profile["dataset"]["rows_per_day_summary"]["median"],
        "rows_per_day_mean": profile["dataset"]["rows_per_day_summary"]["mean"],
    }
])
overview_df


,row_count,unique_topic_count,unique_date_count,rows_per_day_min,rows_per_day_max,rows_per_day_median,rows_per_day_mean
0,101731,33973,749,35,244,136.0,135.82243


## 3. Column Inspection


In [4]:
column_df = pd.DataFrame(
    {
        "column": profile["dataset"]["columns"],
        "dtype": [
            profile["dataset"]["dtypes"][column]
            for column in profile["dataset"]["columns"]
        ],
    }
)
column_df


,column,dtype
0,num_hours,int8
1,date,object
2,name,str
3,counts,float64


## 4. Missing Values And Duplicates


In [5]:
missing_df = pd.DataFrame(
    list(profile["missing_values"].items()),
    columns=["column", "missing_count"],
)

duplicate_df = pd.DataFrame(
    [
        {
            "duplicate_rows": profile["duplicates"]["duplicate_rows"],
            "duplicate_date_name_rows": profile["duplicates"]["duplicate_date_name_rows"],
            "repeated_topics_ge_2": profile["duplicates"]["repeated_topics_ge_2"],
            "repeated_topics_ge_10": profile["duplicates"]["repeated_topics_ge_10"],
        }
    ]
)

missing_df, duplicate_df


(      column  missing_count
 0  num_hours              0
 1       date              0
 2       name              0
 3     counts              0,
    duplicate_rows  duplicate_date_name_rows  repeated_topics_ge_2  \
 0             122                       122                 14119   
 
    repeated_topics_ge_10  
 0                   1884  )

In [6]:
top_topics_df = pd.DataFrame(profile["duplicates"]["top_topics"])
top_topics_df.head(25)


,name,count
0,#OPLive,119
1,#WWERaw,116
2,#SmackDown,109
3,#WWENXT,109
4,Good Tuesday,107
5,Good Monday,107
6,Good Saturday,106
7,#FursuitFriday,106
8,Good Sunday,106
9,Good Friday,105


## 5. Date And Topic Distributions


In [7]:
rows_per_day_df = pd.DataFrame(profile["date_distribution"])
rows_per_day_df["date"] = pd.to_datetime(rows_per_day_df["date"])
rows_per_day_df = rows_per_day_df.sort_values("date").reset_index(drop=True)

rows_per_day_df


,date,row_count
0,2024-01-01,102
1,2024-01-02,122
2,2024-01-03,104
3,2024-01-04,110
4,2024-01-05,118
...,...,...
744,2026-01-20,129
745,2026-01-21,154
746,2026-01-22,148
747,2026-01-23,147


In [8]:
rows_per_day_df["row_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


count    749.000000
mean     135.822430
std       16.092953
min       35.000000
50%      136.000000
90%      154.000000
95%      159.000000
99%      170.520000
max      244.000000
Name: row_count, dtype: float64

## 6. Numeric Outlier Checks


In [9]:
counts_outlier_df = pd.DataFrame([profile["numeric_outliers"]["counts"]])
num_hours_outlier_df = pd.DataFrame([profile["numeric_outliers"]["num_hours"]])

counts_outlier_df, num_hours_outlier_df


(   row_count  iqr_lower  iqr_upper  outlier_count  outlier_ratio  \
 0     101731  -56689.25  107364.75          11999       0.117948   
 
                                             describe  
 0  {'count': 101731.0, 'mean': 64267.73471213298,...  ,
    row_count  iqr_lower  iqr_upper  outlier_count  outlier_ratio  \
 0     101731       -5.0       11.0           2202       0.021645   
 
                                             describe  
 0  {'count': 101731.0, 'mean': 3.4536866835084683...  )

## 7. Topic-Length, Hashtag, And Noise Analysis


In [10]:
text_profile = profile["text_profile"]

topic_length_char_df = pd.DataFrame([text_profile["topic_length_chars"]])
topic_length_token_df = pd.DataFrame([text_profile["topic_length_tokens"]])
noise_metrics_df = pd.DataFrame([text_profile["noise_metrics"]])

topic_length_char_df, topic_length_token_df, noise_metrics_df


(      count      mean       std  min  p50   p90   p95   p99   max
 0  101731.0  9.224936  4.396255  4.0  8.0  15.0  18.0  24.0  30.0,
       count      mean       std  min  p50  p90  p95  p99  max
 0  101731.0  1.400841  0.645748  1.0  1.0  2.0  2.0  4.0  7.0,
    starts_with_hash  contains_ampersand  contains_apostrophe_ascii  \
 0             17180                  83                        343   
 
    contains_apostrophe_curly  contains_hyphen  contains_comma  \
 0                         21              424              30   
 
    contains_period  contains_digits  contains_dollar  contains_non_ascii  \
 0              389             4220             2158                 546   
 
    contains_non_alnum_space_hash  multi_word_rows  
 0                           4406            34032  )

In [11]:
pd.DataFrame(
    [
        {
            "hashtag_rows": text_profile["hashtag_rows"],
            "unique_hashtags": text_profile["unique_hashtags"],
            "case_variant_collisions": text_profile["case_variant_collisions"],
        }
    ]
)


,hashtag_rows,unique_hashtags,case_variant_collisions
0,17180,7032,1283


In [12]:
top_hashtags_df = pd.DataFrame(text_profile["top_hashtags"])
top_hashtags_df.head(25)


,hashtag,count
0,#OPLive,119
1,#WWERaw,116
2,#SmackDown,109
3,#WWENXT,109
4,#FursuitFriday,106
5,#MondayMotivation,104
6,#AEWDynamite,103
7,#FridayVibes,103
8,#Caturday,101
9,#sundayvibes,97


In [13]:
noise_examples_df = pd.DataFrame({"example_noisy_name": text_profile["noise_examples"]})
noise_examples_df


,example_noisy_name
0,#FreenBecky1stFMVN
1,$MEW
2,Big 4
3,Elite 8
4,ELITE 8
5,#80million
6,Flau’jae
7,$BENJI
8,$BOSI
9,$PLANET


## 8. Candidate Normalization Rules


In [14]:
rules = profile["candidate_cleaning_rules"]
for index, rule in enumerate(rules, start=1):
    print(f"{index}. {rule}")


1. Unicode-normalize to NFKC, trim outer whitespace, and collapse repeated internal whitespace.
2. Case-fold to lowercase for matching keys while preserving original display name separately.
3. Build two normalized keys: one with leading # removed and one retaining hashtag semantics.
4. Replace punctuation separators (&, -, apostrophes, periods) with spaces before token matching.
5. Normalize curved apostrophes and similar punctuation to ASCII equivalents before stripping.
6. Strip leading $ from ticker-like terms into a secondary ticker key while keeping original token.
7. Retain alphanumeric tokens; remove other punctuation except # and $ in dedicated variants.
8. Deduplicate by (date, normalized_key_no_hash) during matching prep.
9. Use both exact normalized-key match and fallback fuzzy/semantic tiers for robust recall.


## 9. Matching-Readiness Recommendation


In [15]:
exact_match_only_is_sufficient = False

rationale = {
    "case_variant_collisions": text_profile["case_variant_collisions"],
    "starts_with_hash": text_profile["noise_metrics"]["starts_with_hash"],
    "contains_non_alnum_space_hash": text_profile["noise_metrics"]["contains_non_alnum_space_hash"],
    "contains_digits": text_profile["noise_metrics"]["contains_digits"],
    "contains_dollar": text_profile["noise_metrics"]["contains_dollar"],
}

print("exact_match_only_is_sufficient:", exact_match_only_is_sufficient)
print("rationale:", rationale)
print(
    "recommended_next_phase: implement deterministic normalization keys first, "
    "then use exact-first with fuzzy and semantic fallback in matching"
)


exact_match_only_is_sufficient: False
rationale: {'case_variant_collisions': 1283, 'starts_with_hash': 17180, 'contains_non_alnum_space_hash': 4406, 'contains_digits': 4220, 'contains_dollar': 2158}
recommended_next_phase: implement deterministic normalization keys first, then use exact-first with fuzzy and semantic fallback in matching
